# Model Experiments
Compare various baseline models, ensemble models and boosting models.

In [ ]:
import pandas as pd
import sys
sys.path.append("..")
from src.models.train import train_and_save_model

df = pd.read_csv("../data/processed/cleaned_tickets.csv")
df.head()

## Predict Category

In [ ]:
from src.features.feature_engineering import get_tfidf_vectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Split and Vectorize
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["category"], test_size=0.2, random_state=42)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

vec = get_tfidf_vectorizer()
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced")
}

for name, model in models.items():
    print(f"--- {name} ---")
    model.fit(X_train_vec, y_train_enc)
    y_pred = model.predict(X_test_vec)
    print(classification_report(y_test_enc, y_pred, target_names=le.classes_, zero_division=0))


## Ensemble Learning

In [ ]:
from sklearn.ensemble import VotingClassifier

estimators = [(name, model) for name, model in models.items()]
voting = VotingClassifier(estimators=estimators, voting="hard")
voting.fit(X_train_vec, y_train_enc)
print("Voting Classifier Report:")
print(classification_report(y_test_enc, voting.predict(X_test_vec), target_names=le.classes_, zero_division=0))